In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from xgboost import XGBRegressor

import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt

import joblib
import json
import os
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# Create folders
# ==========================================

os.makedirs('../models', exist_ok=True)
os.makedirs('../plots', exist_ok=True)

# ==========================================
# Load Dataset
# ==========================================

df = pd.read_csv('../data/cc_segmented.csv')

print("Dataset Loaded")
print(df.shape)

# ==========================================
# Encode Categorical Features
# ==========================================

le_city = LabelEncoder()

le_payment = LabelEncoder()

le_persona = LabelEncoder()

df['city_enc'] = le_city.fit_transform(
    df['city_tier']
)

df['payment_enc'] = le_payment.fit_transform(
    df['payment_behavior']
)

df['persona_enc'] = le_persona.fit_transform(
    df['persona']
)

# ==========================================
# Feature Engineering
# ==========================================

df['income_spend_ratio'] = (
    df['monthly_spend'] /
    (df['income_mo'] + 1)
)

df['spend_per_txn'] = (
    df['monthly_spend'] /
    (df['txn_count'] + 1)
)

df['high_util_flag'] = (
    df['util_rate'] > 0.70
).astype(int)

# ==========================================
# Features
# ==========================================

FEATURES = [

    'age',

    'income_mo',

    'cibil',

    'tenure_mo',

    'city_enc',

    'monthly_spend',

    'txn_count',

    'avg_txn_value',

    'util_rate',

    'payment_enc',

    'reward_pts_mo',

    'intl_txn_pct',

    'missed_payments',

    'autopay_enabled',

    'persona_enc',

    'income_spend_ratio',

    'spend_per_txn',

    'high_util_flag'
]

TARGET = 'credit_limit'

X = df[FEATURES]

y = df[TARGET]

# ==========================================
# Train Test Split
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42
)

print()
print(f"Train Shape: {X_train.shape}")

print(f"Test Shape : {X_test.shape}")

# ==========================================
# XGBoost Model (LIGHT VERSION)
# ==========================================

xgb_model = XGBRegressor(

    n_estimators=120,

    max_depth=5,

    learning_rate=0.08,

    subsample=0.8,

    colsample_bytree=0.8,

    eval_metric='rmse',

    random_state=42,

    n_jobs=-1,

    verbosity=0
)

# ==========================================
# Train Model
# ==========================================

print()
print("Training XGBoost Model...")

xgb_model.fit(

    X_train,

    y_train,

    eval_set=[(X_test, y_test)],

    verbose=False
)

# ==========================================
# Predictions
# ==========================================

y_pred = xgb_model.predict(X_test)

# ==========================================
# Metrics
# ==========================================

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

mae = mean_absolute_error(
    y_test,
    y_pred
)

r2 = r2_score(
    y_test,
    y_pred
)

print()
print("MODEL PERFORMANCE")
print("-" * 40)

print(f"RMSE : Rs {rmse:,.0f}")

print(f"MAE  : Rs {mae:,.0f}")

print(f"R²   : {r2:.4f}")

# ==========================================
# Feature Importance
# ==========================================

importance = xgb_model.feature_importances_

feat_imp = pd.DataFrame({

    'feature': FEATURES,

    'importance': importance

}).sort_values(
    by='importance',
    ascending=False
)

print()
print("Top Features:")

print(
    feat_imp.head(10)
)

# ==========================================
# Plot Feature Importance
# ==========================================

plt.figure(figsize=(10,6))

plt.barh(

    feat_imp['feature'][:10][::-1],

    feat_imp['importance'][:10][::-1],

    color='#2563EB'
)

plt.title(
    'Top Feature Importance',
    fontweight='bold'
)

plt.xlabel('Importance Score')

plt.tight_layout()

plt.savefig(

    '../plots/feature_importance.png',

    dpi=100,

    bbox_inches='tight'
)

plt.close()

# ==========================================
# Save Models
# ==========================================

joblib.dump(

    xgb_model,

    '../models/limit_predictor.pkl'
)

joblib.dump(

    le_city,

    '../models/le_city.pkl'
)

joblib.dump(

    le_payment,

    '../models/le_payment.pkl'
)

joblib.dump(

    le_persona,

    '../models/le_persona.pkl'
)

# ==========================================
# Save Metrics
# ==========================================

json.dump(

    {

        'rmse': round(rmse, 0),

        'mae': round(mae, 0),

        'r2': round(r2, 4),

        'features': FEATURES

    },

    open('../models/limit_metrics.json', 'w'),

    indent=2
)

print()
print("Saved:")
print("models/limit_predictor.pkl")
print("models/limit_metrics.json")
print("plots/feature_importance.png")

Dataset Loaded
(30000, 24)

Train Shape: (24000, 18)
Test Shape : (6000, 18)

Training XGBoost Model...

MODEL PERFORMANCE
----------------------------------------
RMSE : Rs 4,944
MAE  : Rs 2,952
R²   : 0.9957

Top Features:
               feature  importance
1            income_mo    0.731573
5        monthly_spend    0.074387
17      high_util_flag    0.046355
8            util_rate    0.038166
14         persona_enc    0.037913
15  income_spend_ratio    0.021721
16       spend_per_txn    0.021022
7        avg_txn_value    0.017612
6            txn_count    0.002173
10       reward_pts_mo    0.002110

Saved:
models/limit_predictor.pkl
models/limit_metrics.json
plots/feature_importance.png
